# RAG Pipeline — Complete End to End
### HuggingFace Embeddings + ChromaDB + Claude LLM + LangChain

---

```
STEP 1  Install packages
STEP 2  Set API Key
STEP 3  Import libraries
STEP 4  Create docs/ folder and write .txt files to disk
STEP 5  Load documents using DirectoryLoader
STEP 6  Split documents into chunks
STEP 7  Load HuggingFace embedding model
STEP 8  Embed chunks and store in ChromaDB
STEP 9  Create retriever from ChromaDB
STEP 10 Set up Claude LLM
STEP 11 Build RAG prompt template
STEP 12 Build full RAG chain
STEP 13 Run queries and get answers
```

---
**Author:** Pawan | IIIT Hyderabad AI/ML Program | Accenture  
**Portfolio:** AI/ML GitHub Portfolio — PM Lens Series

---
## STEP 1 — Install All Packages

In [ ]:
# Run this cell first. It installs everything needed for the RAG pipeline.
# Takes 1-2 minutes on Colab.

!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-anthropic
!pip install -q chromadb
!pip install -q sentence-transformers
!pip install -q anthropic

print("")
print("All packages installed:")
print("  langchain          - orchestration framework")
print("  langchain-community - document loaders, ChromaDB, HuggingFace")
print("  langchain-anthropic - Claude LLM wrapper")
print("  chromadb           - local vector store")
print("  sentence-transformers - HuggingFace embedding model")
print("  anthropic          - Anthropic API client")

---
## STEP 2 — Set Anthropic API Key

**How to add your key in Colab:**
1. Click the **key icon** in the left sidebar
2. Click **Add new secret**
3. Name: `ANTHROPIC_API_KEY`
4. Value: your actual key starting with `sk-ant-...`
5. Toggle **Notebook access** to ON

Get your key from: https://console.anthropic.com

In [ ]:
import os

# Option A — Colab Secrets (RECOMMENDED, key never exposed in notebook)
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("API key loaded from Colab Secrets")

except Exception:
    # Option B — Paste key directly (for local testing only, NEVER commit to GitHub)
    os.environ["ANTHROPIC_API_KEY"] = "sk-ant-paste-your-key-here"
    print("API key set manually — do NOT push this to GitHub")

# Verify key is set
key = os.environ.get("ANTHROPIC_API_KEY", "")
if key.startswith("sk-ant"):
    print(f"Key loaded successfully: sk-ant-...{key[-6:]}")
else:
    print("ERROR: Key not set correctly. Check Step 2.")

---
## STEP 3 — Import All Libraries

In [ ]:
import os
import shutil
import warnings
warnings.filterwarnings("ignore")

# ── Document Loading ──────────────────────────────────────────────────
# DirectoryLoader : scans a folder and loads all matching files
# TextLoader      : reads individual .txt files
from langchain_community.document_loaders import DirectoryLoader, TextLoader

# ── Document Splitting ────────────────────────────────────────────────
# Splits large documents into smaller chunks for embedding
from langchain.text_splitter import RecursiveCharacterTextSplitter

# ── Embeddings ────────────────────────────────────────────────────────
# HuggingFace model that converts text into vectors (numbers)
# Free, runs locally, no API key needed
from langchain_community.embeddings import HuggingFaceEmbeddings

# ── Vector Store ──────────────────────────────────────────────────────
# ChromaDB stores vectors + text, enables semantic search
from langchain_community.vectorstores import Chroma

# ── LLM (Claude) ──────────────────────────────────────────────────────
from langchain_anthropic import ChatAnthropic

# ── Chain Building ────────────────────────────────────────────────────
# ChatPromptTemplate : defines the prompt structure with variables
# RunnablePassthrough: passes input through unchanged in a chain
# StrOutputParser    : extracts plain text from Claude response object
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser

print("All libraries imported successfully")
print()
print("What each import does:")
print("  DirectoryLoader      -> scans docs/ folder, loads all .txt files")
print("  TextLoader           -> reads a single .txt file")
print("  RecursiveCharTextSpl -> splits documents into 500-char chunks")
print("  HuggingFaceEmbeddings-> converts text to 384-dim vector")
print("  Chroma               -> stores and searches vectors")
print("  ChatAnthropic        -> Claude LLM wrapper")
print("  ChatPromptTemplate   -> prompt with {context} and {question} slots")
print("  RunnablePassthrough  -> passes question through unchanged")
print("  StrOutputParser      -> extracts clean text from Claude output")

---
## STEP 4 — Create docs/ Folder and Write Document Files

We create real `.txt` files on disk — this is how production RAG works.
Documents live as actual files, not hardcoded strings in code.

**In your real project:** Skip this cell. Just upload your own `.txt` or `.pdf` files
into the `docs/` folder using the Colab file browser (left sidebar → Files icon).

```
docs/
├── pricing_policy.txt
├── refund_policy.txt
├── sla_policy.txt
├── security_compliance.txt
└── onboarding_support.txt
```

In [ ]:
# Create the docs/ directory
os.makedirs("docs", exist_ok=True)

# ── Document 1: Pricing ─────────────────────────────────────────────
with open("docs/pricing_policy.txt", "w", encoding="utf-8") as f:
    f.write("""PRICING POLICY - StackFlow SaaS Platform

Starter Plan: Rs.999 per month. Up to 5 users.
Includes basic analytics, email support, and standard integrations.

Growth Plan: Rs.2999 per month. Up to 25 users.
Includes advanced analytics, API access, priority support, custom dashboards,
and webhook integrations.

Enterprise Plan: Custom pricing. Unlimited users.
Includes dedicated Customer Success Manager, SLA guarantee, SSO and SAML integration,
on-premise deployment option, and 24x7 phone support.

Annual billing gives a 20 percent discount on all plans.
All plans include a 14-day free trial with no credit card required.
""")

# ── Document 2: Refund Policy ───────────────────────────────────────
with open("docs/refund_policy.txt", "w", encoding="utf-8") as f:
    f.write("""REFUND AND CANCELLATION POLICY - StackFlow SaaS Platform

Monthly plans:
Customers can cancel anytime. No refund for the current billing month.
Access continues until the end of the paid period.

Annual plans:
Customers who cancel within the first 30 days get a full refund.
After 30 days, no refund is issued. The account stays active for the rest of the annual term.

Enterprise customers:
Refund terms are defined in the Master Service Agreement signed at onboarding.
Refund requests must go through the Customer Success Manager within 15 business days of billing.

Downgrade policy:
Downgrading takes effect at the next billing cycle.
No pro-rated refund is issued for unused features.
""")

# ── Document 3: SLA ─────────────────────────────────────────────────
with open("docs/sla_policy.txt", "w", encoding="utf-8") as f:
    f.write("""SERVICE LEVEL AGREEMENT - StackFlow Enterprise Tier

Uptime guarantee: 99.9 percent monthly uptime for Enterprise customers.
Scheduled maintenance is excluded and communicated 72 hours in advance.

Incident response times:
P1 Critical - system completely down: Response in 1 hour, resolution target 4 hours.
P2 High - major feature unavailable: Response in 4 hours, resolution in 24 hours.
P3 Medium - minor degradation: Response within 1 business day.
P4 Low - cosmetic or informational issue: Response within 3 business days.

SLA credits:
If uptime falls below 99.9 percent, customers receive 10 percent credit per 0.1 percent shortfall.
Maximum credit is 30 percent of the monthly invoice, applied to the next billing cycle.
""")

# ── Document 4: Security ────────────────────────────────────────────
with open("docs/security_compliance.txt", "w", encoding="utf-8") as f:
    f.write("""DATA SECURITY AND COMPLIANCE - StackFlow Platform

Encryption:
All data is encrypted at rest using AES-256 and in transit using TLS 1.3.

Data residency:
By default, data is stored in AWS Mumbai ap-south-1 region.
Enterprise customers can request storage in EU Frankfurt or US Virginia.

Compliance:
Certified SOC 2 Type II, ISO 27001, and GDPR compliant.
DPDP Act India compliance achieved in January 2024.

Data retention:
Data is kept for 90 days after cancellation before permanent deletion.
Enterprise customers can request up to 12 months retention under a Data Processing Agreement.

Penetration testing:
Annual third-party pen tests are conducted.
Reports are available to Enterprise customers under NDA.
""")

# ── Document 5: Onboarding ──────────────────────────────────────────
with open("docs/onboarding_support.txt", "w", encoding="utf-8") as f:
    f.write("""ONBOARDING AND SUPPORT - StackFlow Platform

Starter plan:
Self-serve onboarding via in-app guides and the documentation portal.
Email support with 48-hour response time.

Growth plan:
One guided onboarding session of 60 minutes with an onboarding specialist.
Priority email and chat support with 8-hour response time.

Enterprise plan:
Dedicated Customer Success Manager assigned within 2 business days of contract signing.
White-glove onboarding includes data migration, SSO setup, custom workflows, and 5 training sessions.
24x7 phone and dedicated Slack channel support.

All plans:
Access to StackFlow Academy video courses, community forum, and monthly product webinars.
""")

# Verify files created
files = sorted(os.listdir("docs"))
print(f"Created {len(files)} files in docs/ folder:")
print()
for fname in files:
    size = os.path.getsize(f"docs/{fname}")
    print(f"  {fname}  —  {size} bytes")

---
## STEP 5 — Load Documents Using DirectoryLoader

`DirectoryLoader` scans the `docs/` folder and loads every `.txt` file as a
LangChain `Document` object.

Each Document has two parts:
- `page_content` — the full text of the file
- `metadata` — automatically filled with the source filename and path

Why `DirectoryLoader` instead of hardcoded strings?
- Works with real files on disk
- Add or remove files in docs/ without changing any code
- Scales to hundreds of documents
- Same pattern used in production systems

In [ ]:
loader = DirectoryLoader(
    path="docs/",                         # folder to scan
    glob="**/*.txt",                      # load all .txt files (** includes subfolders)
    loader_cls=TextLoader,                # use TextLoader to read each file
    loader_kwargs={"encoding": "utf-8"},  # encoding for reading files
    show_progress=True,                   # show progress bar
    use_multithreading=True               # load files in parallel (faster for large folders)
)

documents = loader.load()

print(f"")
print(f"Loaded {len(documents)} documents")
print()

for i, doc in enumerate(documents):
    filename = os.path.basename(doc.metadata["source"])
    print(f"  Document {i+1}: {filename}")
    print(f"    Characters : {len(doc.page_content)}")
    print(f"    Metadata   : {doc.metadata}")
    print(f"    Preview    : {doc.page_content[:80].strip()}...")
    print()

---
## STEP 6 — Split Documents into Chunks

**Why split?**
- LLMs have token limits — cannot send 500 pages at once
- Smaller chunks retrieve more precisely from ChromaDB
- ChromaDB searches at chunk level, not document level

**How `RecursiveCharacterTextSplitter` works:**
It tries to split on `\n\n` first (paragraph), then `\n` (line), then `.` (sentence), then space.
This preserves meaning better than hard character cuts.

```
Full Document (2000 chars)
         ↓  chunk_size=500, chunk_overlap=50
Chunk 1: chars 0   → 500
Chunk 2: chars 450 → 950    ← 50 char overlap keeps boundary context
Chunk 3: chars 900 → 1400
Chunk 4: chars 1350→ 1850
```

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,                       # max characters per chunk
    chunk_overlap=50,                     # characters shared between consecutive chunks
    separators=["\n\n", "\n", ".", " "]  # split priority: paragraph > line > sentence > word
)

chunks = text_splitter.split_documents(documents)

print("Splitting complete")
print()
print(f"  Documents loaded : {len(documents)}")
print(f"  Total chunks     : {len(chunks)}")
print(f"  Avg chunk size   : {sum(len(c.page_content) for c in chunks) // len(chunks)} chars")
print()
print("Sample chunk (chunk index 2):")
print("-" * 60)
print(f"  Source  : {os.path.basename(chunks[2].metadata['source'])}")
print(f"  Content : {chunks[2].page_content}")
print("-" * 60)

---
## STEP 7 — Load HuggingFace Embedding Model

The embedding model converts text into a list of numbers (a vector).
Texts with similar meaning produce vectors that are close to each other in space.
This is what enables semantic search in ChromaDB.

**Model: `all-MiniLM-L6-v2`**
- Free — no API key
- Runs locally inside Colab — your data never leaves
- Downloads ~90MB once, then cached
- Output: 384 numbers per input text

```
"Refund policy"      → [0.21, -0.83, 0.44, ...]  384 numbers
"Money back"         → [0.20, -0.80, 0.46, ...]  very similar
"Cricket schedule"   → [-0.67, 0.34, -0.21, ...] very different
```

**Critical:** The SAME model must embed both the chunks (at index time)
AND the user question (at query time). Different models = different spaces = wrong results.

In [ ]:
print("Loading HuggingFace embedding model...")
print("(First run downloads ~90MB — subsequent runs use cache)")
print()

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},               # use "cuda" if Colab GPU is enabled
    encode_kwargs={"normalize_embeddings": True}  # normalize vectors for cosine similarity
)

# Quick test: embed a sample sentence and inspect the vector
test_vector = embedding_model.embed_query("What is the refund policy?")

print("Embedding model loaded")
print()
print(f"  Model      : sentence-transformers/all-MiniLM-L6-v2")
print(f"  Dimensions : {len(test_vector)}  (each text becomes {len(test_vector)} numbers)")
print(f"  Sample     : {[round(v, 4) for v in test_vector[:8]]}  ... (first 8 of 384)")

---
## STEP 8 — Embed Chunks and Store in ChromaDB

This is the **indexing phase**. It runs once.

What happens inside `Chroma.from_documents()`:
1. Each chunk is passed through the HuggingFace model → becomes a 384-dim vector
2. ChromaDB stores: vector + original text + metadata (source filename)
3. Everything is saved to `./chroma_db/` on disk

After this step, ChromaDB is your searchable knowledge base.
You can reload it later without re-embedding (see commented code at bottom).

In [ ]:
# Remove previous ChromaDB if exists (avoids duplicate vectors on re-run)
if os.path.exists("./chroma_db"):
    shutil.rmtree("./chroma_db")
    print("Cleared old ChromaDB")

print("Embedding chunks and storing in ChromaDB...")
print()

# This single call does three things:
# 1. Embeds every chunk using HuggingFace model
# 2. Stores (vector + text + metadata) in ChromaDB
# 3. Persists everything to ./chroma_db/ folder on disk
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./chroma_db",
    collection_name="saas_policies"
)

print("ChromaDB indexing complete")
print()
print(f"  Vectors stored   : {vectorstore._collection.count()}")
print(f"  Saved to         : ./chroma_db/")
print(f"  Collection name  : saas_policies")
print()
print("# To reload later without re-embedding:")
print("# vectorstore = Chroma(persist_directory='./chroma_db',")
print("#               embedding_function=embedding_model,")
print("#               collection_name='saas_policies')")

---
## STEP 9 — Create Retriever from ChromaDB

The retriever is the search interface on top of ChromaDB.

When you give it a question, it:
1. Embeds the question using the **same HuggingFace model**
2. Computes **cosine similarity** between question vector and every stored chunk vector
3. Returns the **top-3 most similar chunks**

We test it independently here before wiring it into the full chain.
This is good practice — confirm retrieval works before debugging the LLM.

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",   # cosine similarity search
    search_kwargs={"k": 3}      # return top 3 most relevant chunks
)

print("Retriever created")
print()

# ── Test retriever independently ──────────────────────────────────────
# Good practice: verify retrieval before running the full chain
test_question = "What happens if I cancel my annual plan after 2 months?"

print(f"Test question: {test_question}")
print()

retrieved_chunks = retriever.invoke(test_question)

print(f"ChromaDB returned {len(retrieved_chunks)} chunks:")
print()
for i, chunk in enumerate(retrieved_chunks):
    filename = os.path.basename(chunk.metadata["source"])
    print(f"  Chunk {i+1}")
    print(f"    Source  : {filename}")
    print(f"    Content : {chunk.page_content[:200].strip()}...")
    print()

---
## STEP 10 — Set Up Claude LLM

Claude receives the prompt (context + question) and generates the final answer.

`temperature=0` makes output deterministic — same question always gives same answer.
This is correct for factual Q&A. Use higher temperature only for creative tasks.

In [ ]:
llm = ChatAnthropic(
    model="claude-3-haiku-20240307",           # fast and cheap — good for RAG
    anthropic_api_key=os.environ["ANTHROPIC_API_KEY"],
    temperature=0,                              # 0 = deterministic, consistent answers
    max_tokens=1024                             # max length of Claude's answer
)

print("Claude LLM ready")
print()
print("  Model       : claude-3-haiku-20240307")
print("  Temperature : 0  (deterministic — right for policy Q&A)")
print("  Max tokens  : 1024")
print()
print("Other model options:")
print("  claude-3-5-sonnet-20241022  — better reasoning, higher cost")
print("  claude-3-opus-20240229      — most powerful, most expensive")

---
## STEP 11 — Build the RAG Prompt Template

The prompt is the instruction Claude receives. It has two variable slots:
- `{context}` — filled with the top-3 chunks retrieved from ChromaDB
- `{question}` — filled with the user's question

Two critical instructions that make this RAG (not just LLM):

**Instruction 1:** `Answer ONLY based on the context below`
→ Forces Claude to use YOUR documents, not its general training data
→ Without this, Claude ignores ChromaDB and answers from memory

**Instruction 2:** `If not found in context, say so`
→ Hallucination guard — Claude says "I don't know" instead of making something up
→ Critical for enterprise / production use

In [ ]:
RAG_PROMPT = """You are a helpful assistant for StackFlow SaaS platform.

Answer the user's question ONLY based on the context provided below.
Do NOT use any outside knowledge or information from your training data.

If the answer is not found in the context, say exactly:
"I don't have enough information in the provided documents to answer this question."

Be concise, specific, and factual.

Context (retrieved from documents):
{context}

Question: {question}

Answer:"""

prompt_template = ChatPromptTemplate.from_template(RAG_PROMPT)

print("Prompt template created")
print()
print("  {context}  <- will be filled by ChromaDB top-3 chunks")
print("  {question} <- will be filled by user's question")
print()
print("Key instructions in prompt:")
print("  1. ONLY use context  -> prevents Claude using general knowledge")
print("  2. Say so if missing -> hallucination guard")

---
## STEP 12 — Build the Full RAG Chain

This wires everything together using LangChain's pipe (`|`) syntax called LCEL.

```
                     user question
                          │
           ┌──────────────┴──────────────┐
           │                             │
      retriever                 RunnablePassthrough
           │                             │
      format_docs                   {question}
           │                             │
       {context}                         │
           │                             │
           └──────────────┬──────────────┘
                          │
                   prompt_template
                  (context + question
                   filled into prompt)
                          │
                      Claude LLM
                          │
                   StrOutputParser
                          │
                     final answer
```

`format_docs` combines the 3 retrieved chunks into one string with source labels,
so Claude can see which file each piece of context came from.

In [ ]:
def format_docs(docs):
    """
    Takes the list of retrieved chunks from ChromaDB.
    Combines them into one string with source file labels.
    This becomes the {context} variable in the prompt.

    Output looks like:
    [Source: refund_policy.txt]
    Annual plans: Customers who cancel within...

    ---

    [Source: pricing_policy.txt]
    Enterprise Plan: Custom pricing...
    """
    formatted = []
    for doc in docs:
        filename = os.path.basename(doc.metadata.get("source", "unknown"))
        formatted.append(f"[Source: {filename}]\n{doc.page_content.strip()}")
    return "\n\n---\n\n".join(formatted)


# ── Build the RAG chain using LCEL pipe syntax ──────────────────────
rag_chain = (
    {
        "context" : retriever | format_docs,  # retriever fetches chunks, format_docs formats them
        "question": RunnablePassthrough()      # user question passed through unchanged
    }
    | prompt_template    # context + question injected into the prompt
    | llm                # filled prompt sent to Claude
    | StrOutputParser()  # Claude response object converted to plain string
)

print("RAG chain built successfully")
print()
print("Chain flow:")
print("  user_question")
print("      -> retriever (ChromaDB semantic search)")
print("      -> format_docs (combine chunks with source labels)")
print("      -> prompt_template (fill {context} and {question})")
print("      -> Claude LLM (generate answer)")
print("      -> StrOutputParser (extract plain text)")
print("      -> final answer")

---
## STEP 13 — Run Queries and Get Final Answers

The `ask()` function shows the full trace:
- Which chunks ChromaDB retrieved
- Which source files they came from
- Claude's final answer based only on those chunks

In [ ]:
def ask(question: str):
    """
    Runs a question through the complete RAG pipeline.
    Shows retrieval trace and final Claude answer.
    """
    print("=" * 70)
    print(f"QUESTION: {question}")
    print("-" * 70)

    # Step A: Show what ChromaDB retrieved
    retrieved = retriever.invoke(question)
    print(f"RETRIEVED {len(retrieved)} CHUNKS FROM CHROMADB:")
    for i, doc in enumerate(retrieved):
        filename = os.path.basename(doc.metadata.get("source", "unknown"))
        print(f"  [{i+1}] {filename}")
        print(f"       {doc.page_content[:120].strip()}...")
    print("-" * 70)

    # Step B: Run full chain (retriever + prompt + Claude)
    answer = rag_chain.invoke(question)

    print("CLAUDE ANSWER (grounded in your documents):")
    print(answer)
    print("=" * 70)
    print()

In [ ]:
# ── Query 1: Refund — enterprise ─────────────────────────────────────
ask("What is the refund policy for enterprise customers?")

In [ ]:
# ── Query 2: SLA — critical outage ───────────────────────────────────
ask("The platform is completely down. How quickly will the team respond?")

In [ ]:
# ── Query 3: Plan comparison ─────────────────────────────────────────
ask("What is the difference between the Growth plan and the Enterprise plan?")

In [ ]:
# ── Query 4: Security ────────────────────────────────────────────────
ask("Is the platform GDPR compliant and where is the data stored?")

In [ ]:
# ── Query 5: Annual plan cancellation ───────────────────────────────
ask("I signed up for an annual plan 45 days ago. Can I get a refund?")

In [ ]:
# ── Query 6: Hallucination guard test (not in any document) ─────────
# This tests whether Claude correctly says it does not know
# instead of making something up
ask("What is the Android app download link?")

---
## BONUS — Inspect ChromaDB Internals

Peek inside the vector database to see what is actually stored.

In [ ]:
collection = vectorstore._collection

sample = collection.get(
    limit=1,
    include=["documents", "embeddings", "metadatas"]
)

print("ChromaDB Internal Record:")
print()
print(f"  Total vectors stored : {collection.count()}")
print()
print(f"  Source file    : {sample['metadatas'][0].get('source')}")
print(f"  Text stored    : {sample['documents'][0][:150]}...")
print(f"  Vector dims    : {len(sample['embeddings'][0])}")
print(f"  Vector sample  : {[round(v, 4) for v in sample['embeddings'][0][:8]]}  ...(first 8 of 384)")

---
## BONUS — Swap to Your Own Real Documents

To use this with your actual files:
1. Upload files to the `docs/` folder (Colab → Files icon → Upload)
2. Pick the right loader below
3. Re-run Steps 5 through 12 — everything else stays the same

In [ ]:
# ── Your own .txt files ──────────────────────────────────────────────
# Already set up — just replace files in docs/ folder

# ── PDF files from a folder ──────────────────────────────────────────
# !pip install pypdf
# from langchain_community.document_loaders import PyPDFDirectoryLoader
# loader = PyPDFDirectoryLoader("docs/")
# documents = loader.load()

# ── Single PDF file ──────────────────────────────────────────────────
# from langchain_community.document_loaders import PyPDFLoader
# loader = PyPDFLoader("docs/your_contract.pdf")
# documents = loader.load()

# ── Mix of PDF and TXT files ─────────────────────────────────────────
# from langchain_community.document_loaders import PyPDFLoader
# txt_loader = DirectoryLoader("docs/", glob="**/*.txt", loader_cls=TextLoader)
# pdf_loader = DirectoryLoader("docs/", glob="**/*.pdf", loader_cls=PyPDFLoader)
# documents = txt_loader.load() + pdf_loader.load()

# ── Google Drive PDF (Colab only) ────────────────────────────────────
# from google.colab import drive
# from langchain_community.document_loaders import PyPDFLoader
# drive.mount("/content/drive")
# loader = PyPDFLoader("/content/drive/MyDrive/your_doc.pdf")
# documents = loader.load()

# ── Confluence or any web page ───────────────────────────────────────
# from langchain_community.document_loaders import WebBaseLoader
# loader = WebBaseLoader("https://your-confluence-page.atlassian.net/wiki/...")
# documents = loader.load()

print("Uncomment the loader matching your file type.")
print("Then re-run Steps 5 through 12 to re-index into ChromaDB.")

---

## Full Pipeline Summary

```
STEP 4   docs/*.txt files written to disk
             ↓
STEP 5   DirectoryLoader + TextLoader
         → LangChain Documents (page_content + metadata)
             ↓
STEP 6   RecursiveCharacterTextSplitter (chunk_size=500, overlap=50)
         → Chunks (smaller pieces of each document)
             ↓
STEP 7   HuggingFace all-MiniLM-L6-v2
         → 384-dimensional vector per chunk
             ↓
STEP 8   Chroma.from_documents()
         → Vectors + text + metadata stored in ./chroma_db/
             ↓
STEP 9   vectorstore.as_retriever(k=3)
         → On query: embeds question, cosine similarity, returns top-3 chunks
             ↓
STEP 10  ChatAnthropic (claude-3-haiku, temperature=0)
             ↓
STEP 11  ChatPromptTemplate
         "Answer ONLY from context: {context}. Question: {question}"
             ↓
STEP 12  RAG Chain = retriever | format_docs | prompt | Claude | StrOutputParser
             ↓
STEP 13  rag_chain.invoke(question)
         → Final answer grounded in your documents
```

---

## PM Lens — Key Decisions

| Decision | Config | Why |
|---|---|---|
| DirectoryLoader | scans docs/ folder | works with real files, no hardcoding |
| chunk_size=500 | 500 chars per chunk | larger = noisy, smaller = loses context |
| chunk_overlap=50 | 50 char shared | preserves context at boundaries |
| all-MiniLM-L6-v2 | HuggingFace, free | vs OpenAI: free + private, local |
| ChromaDB local | saved to ./chroma_db | vs Pinecone: local=POC, Pinecone=prod |
| k=3 | top 3 chunks | precision vs recall tradeoff |
| temperature=0 | deterministic output | policy Q&A needs facts not creativity |
| Prompt guard | ONLY from context | prevents hallucination |

---
*Built as part of AI/ML GitHub Portfolio — PM Lens Series*  
*Author: Pawan | IIIT Hyderabad AI/ML Program | Accenture*